# Phase 4: Candidate Blocking Evaluation

**Amazon ML Challenge 2026 — Business Entity Resolution**

This notebook benchmarks and evaluates modular candidate blocking strategies on training data against ground truth.

### Phase 4 Objectives:
1. **Tradeoff Optimization**: Balance candidate recall, reduction ratio, candidate volume, and memory.
2. **Independent Benchmarking**: Evaluate exact-name, token, prefix, conservative address, and country-scoped blocking strategies.
3. **Frequency Guards**: Suppress pathological high-frequency blocking keys to prevent memory explosion.
4. **Composite Union**: Combine multiple orthogonal blocks to ensure zero-loss recall on tricky matches.
5. **Ground Truth Validation**: Calculate recall against authoritative training ground truth.

## 1. Setup & Configuration

In [ ]:
import os
import sys
from pathlib import Path

# Ensure repository root is on sys.path
repo_root = Path.cwd()
if not (repo_root / "src").exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import pandas as pd
from src.blocking.config import BlockingConfig
from src.blocking.evaluator import BlockingEvaluator
from src.blocking.strategies import create_default_strategies
from src.data.data_source import LocalDataSource, create_data_source
from src.utils.logger import setup_logger

logger = setup_logger(name="blocking_notebook")
print(f"Repository root: {repo_root}")

## 2. Load Data Source & Ground Truth

In [ ]:
# Local sample data or S3 bucket path
data_path = repo_root / "tests" / "fixtures" / "sample_data"
norm_source = LocalDataSource(base_dir=data_path)
raw_source = LocalDataSource(base_dir=data_path)

config = BlockingConfig(
    max_posting_list_size=500,
    country_agreement_mode="allow_missing",
    seed=42
)

evaluator = BlockingEvaluator(
    norm_source=norm_source,
    raw_source=raw_source,
    config=config,
    logger_instance=logger
)

gt_positives, gt_matches, s1_ids = evaluator.load_ground_truth("train/train_ground_truth.tsv")
print(f"Ground truth loaded: {len(s1_ids)} S1 records, {len(gt_positives)} true positive pairs")

## 3. Run Strategy Benchmark

Benchmark individual strategy families independently and evaluate the composite union.

In [ ]:
benchmark_out = evaluator.run_benchmark(
    strategy_names=["exact_name", "name_token", "name_prefix", "address_conservative", "country_scoped_name", "composite"],
    s1_rel_path="train/train_source1.tsv",
    s2_rel_path="train/train_source2.tsv",
    s3_rel_path="train/train_source3.tsv",
    gt_rel_path="train/train_ground_truth.tsv"
)

report = benchmark_out["report"]
summary_df = pd.DataFrame(report["summary_comparison"])
summary_df

## 4. Recall vs. Candidate Reduction Tradeoff Analysis

In [ ]:
print("=== Strategy Comparison Summary ===")
for idx, row in summary_df.iterrows():
    print(f"Strategy: {row['strategy']:<24} | Recall: {row['recall']:.2%} | Reduction: {row['reduction_ratio']:.4%} | Candidates: {row['candidate_pairs']} | Avg/S1: {row['avg_cands_per_s1']:.2f}")

## 5. False Negative Error Analysis

Inspect missed true positive pairs across strategies to identify complementary blocking signals.

In [ ]:
fn_samples = report.get("false_negative_samples", {})
for strat, missed_list in fn_samples.items():
    print(f"Strategy '{strat}' missed {len(missed_list)} sample pairs:")
    for m in missed_list[:5]:
        print(f"  - S1: {m['source1_id']} -> Target: {m['target_id']} ({m['target_source']})")

## 6. Generated Candidates Inspection

In [ ]:
cands = benchmark_out["candidates"]
cands_df = pd.DataFrame([c.to_dict() for c in cands])
print(f"Total composite candidates generated: {len(cands_df)}")
cands_df.head(10)